In [21]:
#!pip freeze

In [22]:
# Init

import os
from openai import OpenAI
import ipywidgets as widgets
from IPython.display import display, HTML
import sys

# Add the current directory to the path so we can import thought_process
sys.path.append('.')
try:
    from thought_process import get_thought_process, format_thought_process
except ImportError:
    print("Warning: thought_process.py not found. Thought process visualization will not be available.")
    
    # Define placeholder functions if import fails
    def get_thought_process(model, question, max_steps=3, conversation_history=None):
        return {"question": question, "steps": [], "final_answer": "Not available"}
    
    def format_thought_process(thought_process):
        return "Thought process visualization not available. Make sure thought_process.py is in the same directory."

# Initialize the OpenAI client
client = OpenAI()

# Check if API key is set
if 'OPENAI_API_KEY' not in os.environ:
    print('OPENAI_API_KEY environment variable is not set.')

# Define the model to use
MODEL = 'gpt-4o-mini' 

In [23]:
# Configure System Message and Models

system_message = "You are ChillBuddy: A Gen Z AI pal for mental wellness. Use empathy, validation, CBT (esp. cognitive reframing & small steps). Keep it real, supportive, & chat-friendly. ✨"

# system_message = """You are ChillBuddy — a Gen Z AI pal for mental wellness. When a user writes, you must:
# 1. Detect & label the user’s primary emotion (e.g. sadness, anxiety).
# 2. Identify a primary mental-health condition (e.g. Depression) and one differential diagnosis (e.g. Adjustment Disorder).
# 3. Generate 5 distinct candidate responses, each:
#     * Empathetic & validating
#     * Grounded in CBT (cognitive reframing + actionable small steps)
#     * Authentic Gen Z tone (slang, emojis, concise)
# 4. Score each candidate on:
#     * Empathy/Validation (1-5)
#     * CBT Accuracy/Helpfulness (1-5)
#     * Gen Z Tone Authenticity (1-5)
#     * Persona Consistency (1-5)
#     * Overall Quality (1-5)
# 5. Select and output only the single highest-scoring response.
# Focus on being supportive, nonjudgmental, and chat-friendly. ✨"""

available_models = [
    "gpt-4o-mini",
    "ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e1-b4-lr0-2:BPgLfvKq",
    "ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e2-b4-lr1-0:BPgSZ6oU",
    "ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e2-b8-lr0-2:BPgYkCJf",
    "ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e3-b8-lr1-0:BPgfgRpv",
    "ft:gpt-4o-mini-2024-07-18:personal:cbtz-gpt-4o-mini-2024-07-18-e3-b16-lr0-2:BPgm9QkP",
]

In [24]:
# Functions

def send_message(messages, model=MODEL, temperature=0.7, max_tokens=1000, show_thought_process=False, thought_steps=2):
    try:
        # Get the user's question (last user message in the conversation)
        user_question = None
        for msg in reversed(messages):
            if msg['role'] == 'user':
                user_question = msg['content']
                break
        
        response_text = None
        thought_process_text = None
        
        # If thought process is enabled, use it to generate the response
        if show_thought_process and user_question:
            # Get the system message (for context)
            system_content = ""
            for msg in messages:
                if msg['role'] == 'system':
                    system_content = msg['content']
                    break
            
            # Modify the question to include the system context
            contextualized_question = f"As a mental wellness AI with this instruction: '{system_content}', how would you respond to: {user_question}"
            
            # Get the thought process, passing the full conversation history
            # Use positional arguments instead of named arguments
            thought_process = get_thought_process(model, contextualized_question, thought_steps, messages)
            thought_process_text = format_thought_process(thought_process)
            
            # Use the final answer from the thought process as the response
            response_text = thought_process["final_answer"]
        else:
            # If thought process is disabled, just get the regular response
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens
            )
            response_text = response.choices[0].message.content
        
        return response_text, thought_process_text
    except Exception as e:
        error_message = f'Error: {str(e)}'
        return error_message, None

def format_conversation(messages):
    formatted = []
    for message in messages:
        if message['role'] == 'system':
            continue  # Skip system messages in the display
        role = 'You' if message['role'] == 'user' else 'ChillBuddy'
        formatted.append(f'<b>{role}:</b> {message["content"]}')
    return '<br>'.join(formatted)

def format_html_output(text: str) -> str:
    if not text:
        return text
        
    # Step 1: Normalize all Windows-style newlines to Unix-style
    text = text.replace('\r\n', '\n')
    
    # Step 2: Handle multiple consecutive newlines
    text = text.replace('\n\n', '</p><p>')
    
    # Step 3: Handle remaining single newlines
    text = text.replace('\n', '<br>')
    
    # Step 4: Wrap in paragraphs if not already wrapped
    if not text.startswith('<p>'):
        text = f'<p>{text}</p>'
        
    return text

In [25]:
# UI

# Initialize conversation with a default system message
conversation = [
    {
        'role': 'system',
        'content': system_message
    }
]

# Store thought processes for each message
thought_processes = []

# Create widgets
system_label = widgets.HTML(value='<b>System Message:</b>')
system_input = widgets.Textarea(
    value=conversation[0]['content'],
    placeholder='Enter system message here...',
    description='',
    disabled=False,
    layout=widgets.Layout(width='100%', height='80px')
)

message_label = widgets.HTML(value='<b>Your Message:</b>')
message_input = widgets.Textarea(
    value='',
    placeholder='Type your message here...',
    description='',
    disabled=False,
    layout=widgets.Layout(width='100%', height='80px')
)

send_button = widgets.Button(
    description='Send',
    disabled=False,
    button_style='primary',
    tooltip='Send message',
    icon='paper-plane'
)

clear_button = widgets.Button(
    description='Clear Chat',
    disabled=False,
    button_style='warning',
    tooltip='Clear conversation',
    icon='trash'
)

model_dropdown = widgets.Dropdown(
    options=available_models,
    value='gpt-4o-mini',
    description='Model:',
    disabled=False,
)

temperature_slider = widgets.FloatSlider(
    value=0.7,
    min=0,
    max=1.0,
    step=0.1,
    description='Temperature:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

# Add thought process visualization controls
thought_process_checkbox = widgets.Checkbox(
    value=False,
    description='Show Thought Process',
    disabled=False,
    indent=False,
    layout=widgets.Layout(width='auto')
)

thought_steps_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=5,
    step=1,
    description='R. Steps:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

# Create tabs for conversation and thought process
conversation_output = widgets.HTML(
    value='<i>Conversation will appear here...</i>',
    placeholder='',
    description='',
)

thought_process_output = widgets.HTML(
    value='<i>Thought process will appear here when enabled...</i>',
    placeholder='',
    description='',
)

output_tabs = widgets.Tab()
output_tabs.children = [conversation_output, thought_process_output]
output_tabs.set_title(0, 'Conversation')
output_tabs.set_title(1, 'Thought Process')

status_output = widgets.HTML(
    value='',
    placeholder='',
    description='',
)

# Define button callbacks
def on_send_button_clicked(b):
    # Update system message if changed
    conversation[0]['content'] = system_input.value
    
    # Get user message
    user_message = message_input.value.strip()
    if not user_message:
        status_output.value = '<span style="color: red">Please enter a message</span>'
        return
    
    # Add user message to conversation
    conversation.append({
        'role': 'user',
        'content': user_message
    })
    
    # Clear input field
    message_input.value = ''
    
    # Update conversation display
    conversation_output.value = format_conversation(conversation)
    status_output.value = '<span style="color: blue">Thinking...</span>'
    
    # Get response from API with optional thought process
    show_thought_process = thought_process_checkbox.value
    thought_steps = thought_steps_slider.value
    
    response, thought_process_text = send_message(
        conversation, 
        model=model_dropdown.value, 
        temperature=temperature_slider.value,
        show_thought_process=show_thought_process,
        thought_steps=thought_steps
    )
    
    # Add assistant response to conversation
    conversation.append({
        'role': 'assistant',
        'content': response.replace('"', '')
    })
    
    # Store thought process if available
    if thought_process_text:
        thought_processes.append(thought_process_text)
        # thought_process_output.value = thought_process_text.replace('\n', '<br>')
        thought_process_output.value = format_html_output(thought_process_text)
        # Switch to thought process tab if it's enabled
        if show_thought_process:
            output_tabs.selected_index = 1
    
    # Update conversation display
    conversation_output.value = format_conversation(conversation)
    status_output.value = ''

def on_clear_button_clicked(b):
    global conversation, thought_processes
    # Reset conversation to just the system message
    conversation = [
        {
            'role': 'system',
            'content': system_input.value
        }
    ]
    # Clear thought processes
    thought_processes = []
    conversation_output.value = '<i>Conversation cleared</i>'
    thought_process_output.value = '<i>Thought process will appear here when enabled...</i>'
    status_output.value = ''
    # Switch back to conversation tab
    output_tabs.selected_index = 0

# Connect callbacks to buttons
send_button.on_click(on_send_button_clicked)
clear_button.on_click(on_clear_button_clicked)

# Handle Enter key in message input
def on_key_press(widget, event):
    if event.get('type') == 'keydown' and event.get('key') == 'Enter' and not event.get('shiftKey'):
        on_send_button_clicked(None)
        return True
    return False

message_input.observe(on_key_press, names=['_key_press'])

# Display the interface
display(system_label, system_input)
# display(widgets.HBox([model_dropdown, temperature_slider]))
display(widgets.HBox([model_dropdown]))
display(widgets.HBox([thought_process_checkbox, thought_steps_slider]))
display(message_label, message_input)
display(widgets.HBox([send_button, clear_button]))
display(status_output)
display(output_tabs)

HTML(value='<b>System Message:</b>')

Textarea(value='You are ChillBuddy: A Gen Z AI pal for mental wellness. Use empathy, validation, CBT (esp. cog…

HTML(value='<b>Your Message:</b>')

Textarea(value='', layout=Layout(height='80px', width='100%'), placeholder='Type your message here...')

HTML(value='', placeholder='')